# MedDial on Colab: BigQuery in, GPU extraction out

The CSV distribution of MIMIC-III is tens of gigabytes, which a Colab runtime will not hold, and
reference extraction is bound by how fast a local model can read a discharge summary. This notebook
resolves both: MIMIC-III is read from `physionet-data` on BigQuery, and the extractor runs on the
runtime's GPU.

**Before you start you need all three of these.** Any one missing and the run stops at step 3.

1. PhysioNet credentialing for MIMIC-III, with the *Google BigQuery* access request approved for the
   same Google account you will sign into below (PhysioNet account settings -> Cloud).
2. A Google Cloud project of your own. BigQuery bills the account that runs the query, not the one
   that publishes the data. Reading the six tables costs a few GB against the monthly free tier.
3. A GPU runtime: **Runtime -> Change runtime type -> T4 / L4 / A100**.

**What leaves this runtime: nothing.** The model server runs inside the runtime and is reached over
loopback, which is what decision D2 / GOV-3 requires -- MIMIC-derived text is never sent to a hosted
API. Reading MIMIC-III *from* BigQuery is the same direction as downloading the CSVs and does not
touch that rule. The output written under `/content` is derived from restricted data: a Colab runtime
is ephemeral, so decide deliberately where it goes at the end, and do not mount Drive by reflex.


## 0. Confirm the GPU


In [ ]:
!nvidia-smi

# No output above means the runtime has no GPU: Runtime -> Change runtime type -> T4 / L4 / A100.
# Extraction will still run on CPU, at roughly an order of magnitude less throughput.


## 1. Serve a model on the GPU

Ollama is installed into the runtime and left listening on `localhost:11434`. The provider layer
refuses to send restricted clinical text anywhere that is not loopback, and checks that *before* it
opens a socket, so this is the only shape of model server the pipeline accepts.


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import os, subprocess, time
import httpx

# OLLAMA_CONTEXT_LENGTH is the reason this is not a bare `ollama serve`. Ollama
# serves with a 4,096-token context by default, and an over-long prompt is not
# refused -- it is silently truncated from the front. At that size the model
# answers about a note it was never shown in full, and returns something empty
# or unparseable.
#
# What has to fit in one call: the longest concatenated discharge documentation
# in the cohort is 27,717 characters (~7,000 tokens), the JSON schema is ~1,200,
# and the answer needs whatever a note full of entities costs to write out. On a
# card with room the window is doubled rather than shaved.
try:
    import torch
    _gib = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
except ImportError:
    _gib = 0

CONTEXT = 32768 if _gib >= 38 else 16384
os.environ['OLLAMA_CONTEXT_LENGTH'] = str(CONTEXT)

# Half the window for the answer, half for the note and the schema. A truncated
# answer is unparseable, and on a 200-case run an 8,192-token budget was what
# most of the long notes died on -- every failure over ~10,000 characters.
os.environ['MEDDIAL_MAX_TOKENS'] = str(CONTEXT // 2)
print(f'{_gib:.0f} GiB of VRAM -> {CONTEXT}-token window, {CONTEXT // 2}-token answer budget')

subprocess.Popen(
    ['ollama', 'serve'],
    env={**os.environ},
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for _ in range(60):
    try:
        httpx.get('http://localhost:11434/api/tags', timeout=2.0).raise_for_status()
        print('ollama is serving')
        break
    except Exception:
        time.sleep(1.0)
else:
    raise RuntimeError('ollama did not come up; re-run this cell')


### Pick the largest extractor the GPU actually holds

Implementation Plan 12.4 asks for the largest extractor the hardware allows, because extraction error
propagates into every downstream metric. What it must not do is exceed VRAM: a model that swaps is
not a slow run, it is a stalled one. The tag below is chosen from the card Colab handed you.


In [ ]:
# torch is preinstalled on Colab and is not a dependency of this package, so a
# runtime without it is assumed to have no GPU rather than failing here.
try:
    import torch
    gib = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
except ImportError:
    gib = 0

# What has to fit is weights + the KV cache for a 16,384-token window, and the
# card's total is not all available. Rough Q4 weights: 7B ~4.5 GiB, 14B ~9 GiB,
# 32B ~20 GiB, 35B ~24 GiB. A 27B or 32B does NOT fit a 15 GiB T4 -- Ollama will silently run
# part of it on CPU, and 200 notes then take a day instead of an hour.
if gib >= 70:      EXTRACTOR = 'qwen3.5:35b'   # A100/H100 80GB -- the project default
elif gib >= 38:    EXTRACTOR = 'qwen2.5:32b'   # A100 40GB
elif gib >= 21:    EXTRACTOR = 'qwen2.5:14b'   # L4 24GB
elif gib >= 14:    EXTRACTOR = 'qwen2.5:14b'   # T4 16GB, tight but fits at Q4
else:              EXTRACTOR = 'qwen2.5:7b'    # CPU or a small card

# Override here if you want a different tag, but check the residency cell below
# before starting a long run.

os.environ['MEDDIAL_EXTRACTOR'] = EXTRACTOR
print(f'{gib:.0f} GiB of VRAM -> {EXTRACTOR}')


### Check the tag exists before downloading it

Ollama's published tags are not something to write down once and trust. `qwen3.5:32b` is what the
Implementation Plan names and it is not published; `qwen3.5:35b-instruct-q8_0` reads like it should
exist and does not. A wrong tag costs a runtime and only fails at the end of a download, so the
registry is asked first.

In [ ]:
import httpx


def is_published(ref: str) -> bool | None:
    """Whether Ollama publishes ``ref``. ``None`` when the registry is unreachable.

    Asks for the manifest, because the registry does not implement the
    Docker-style `tags/list` endpoint -- that path 404s even for models that
    plainly exist, so it cannot tell "no such tag" from "no such endpoint".
    A manifest 200/404 is unambiguous.
    """
    family, _, tag = ref.partition(':')
    url = f'https://registry.ollama.ai/v2/library/{family}/manifests/{tag or "latest"}'
    try:
        return httpx.get(url, timeout=30.0).status_code == 200
    except Exception as exc:
        print(f'Could not reach the Ollama registry ({type(exc).__name__}: {exc}).')
        return None


# Tags are not stable facts to write down once. qwen3.5:32b is what the
# Implementation Plan names and is not published; qwen3.5:35b-instruct-q8_0 reads
# like it should exist and does not. Checking costs a second; guessing costs a
# runtime, at the end of a model download.
published = is_published(EXTRACTOR)
if published is False:
    raise SystemExit(
        f'{EXTRACTOR!r} is not published by Ollama. Browse https://ollama.com/library/'
        f'{EXTRACTOR.partition(":")[0]}/tags for the sizes that exist, set EXTRACTOR in '
        'the cell above, and re-run from there.'
    )
print(f'{EXTRACTOR}: {"published" if published else "unverified"}')

In [ ]:
!ollama pull $MEDDIAL_EXTRACTOR


### Confirm the window before committing to a long run

`ollama ps` prints the context the model is actually loaded with. It must read 16384, not 4096: a
server that ignored the setting truncates every note silently, and the first sign of that is an
extraction with no symptoms, no diagnoses and no treatments.

Raising it on the server rather than baking it into a derived model keeps the run pinned to the tag
the weights came from. A local `ollama create` copy has its own digest, which identifies a manifest
but names nothing upstream -- and provenance is the point of recording a digest at all (C8).

In [ ]:
# Load the model so it appears in `ollama ps`. One token, and no clinical text.
httpx.post(
    'http://localhost:11434/api/generate',
    json={'model': os.environ['MEDDIAL_EXTRACTOR'], 'prompt': 'hi', 'options': {'num_predict': 1}},
    timeout=600.0,
).raise_for_status()

resident = httpx.get('http://localhost:11434/api/ps', timeout=60.0).json().get('models', [])
for entry in resident:
    ctx = entry.get('context_length')
    size, vram = entry.get('size') or 0, entry.get('size_vram') or 0
    on_gpu = vram / size if size else 0.0
    print(f"{entry.get('name')}: context {ctx}, {on_gpu:.0%} of {size / 1024**3:.1f} GiB on GPU")
    if ctx and int(ctx) < CONTEXT:
        print(f"  !! context is {ctx}, not {CONTEXT}. A note longer than that is truncated in")
        print("     silence, and the first sign of it is an extraction with nothing in it.")
        print("     Restart the runtime and re-run the serve cell.")
    if on_gpu < 0.999:
        print(f"  !! only {on_gpu:.0%} of the weights are on the GPU; the rest run on CPU.")
        print("     This model is too big for this card. A model that swaps is not a slow")
        print("     run, it is a stalled one -- drop to a smaller tag above.")


## 2. Install MedDial

The `bigquery` extra adds the BigQuery client; without it the CSV path still works and `--bigquery`
fails with an instruction to install it.

What is deliberately *not* installed is the `eval` extra -- DeepEval, transformers,
sentence-transformers and the rest of the scoring stack. `meddial-cohort` and `meddial-scr` import
none of it, and installing it here would do more than waste several gigabytes: this image ships
`huggingface-hub` 1.x for its own `gradio` and `diffusers`, and pulling in `transformers` 4.x drags
the hub back to 0.x and breaks both. Add `[eval]` only when you get as far as scoring, and expect to
restart the runtime if you do.


In [ ]:
import os, subprocess

REPO = '/content/FinalProject-MedDial'
URL = 'https://github.com/alongott15/FinalProject-MedDial.git'
BRANCH = 'main'


def git(*args: str) -> str:
    done = subprocess.run(['git', '-C', REPO, *args], capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{done.stderr.strip()}")
    return done.stdout.strip()


# A Colab runtime outlives a cell, and `git clone` into an existing directory
# fails rather than updating it. Left alone, a session keeps running whatever it
# cloned hours ago -- which is exactly how a fix that is already pushed appears
# not to work. So this resets to origin every run.
if os.path.isdir(f'{REPO}/.git'):
    local = git('status', '--porcelain')
    if local:
        print('Discarding local changes in the clone:')
        print(local)
        print('(edit the repository and push instead; this cell always resets to '
              f'origin/{BRANCH}.)\n')
    git('fetch', '--quiet', 'origin', BRANCH)
    git('reset', '--hard', '--quiet', f'origin/{BRANCH}')
    git('clean', '-qfd')
else:
    done = subprocess.run(
        ['git', 'clone', '--quiet', '--branch', BRANCH, URL, REPO],
        capture_output=True, text=True,
    )
    if done.returncode:
        raise RuntimeError(f'git clone failed:\n{done.stderr.strip()}')

print('now at', git('log', '-1', '--pretty=%h %s'))

In [ ]:
%pip install -q -e '/content/FinalProject-MedDial[bigquery]'

## 3. Sign in, and prove the access works before spending an hour on it

The cell below takes your project id from a Colab secret, an environment variable, or a prompt --
never from a literal, which the next re-pull would overwrite with a placeholder.

**Run the pre-flight query too.** It reads one small table, so a Google account whose PhysioNet
credentialing is not linked, or a project that cannot run a job, fails here in a second rather than
part-way through building the cohort.


In [ ]:
from google.colab import auth

auth.authenticate_user()

# Deliberately not a literal in this cell. The clone resets to origin on every
# run and this notebook is re-opened from the repository, so an edited-in project
# id gets replaced by the placeholder -- and the placeholder gets a long way in
# before it fails, because reading table metadata needs no billing project. The
# snapshot line prints, and only the first query fails, minutes later.
#
# Store it once instead: left sidebar -> key icon -> add a secret named
# MIMIC_BIGQUERY_PROJECT, and give this notebook access to it.
PROJECT = os.environ.get('MIMIC_BIGQUERY_PROJECT', '').strip()

if not PROJECT:
    try:
        from google.colab import userdata

        PROJECT = (userdata.get('MIMIC_BIGQUERY_PROJECT') or '').strip()
    except Exception:
        PROJECT = ''

if not PROJECT:
    PROJECT = input('Google Cloud project id (BigQuery bills this project): ').strip()

os.environ['MIMIC_BIGQUERY_PROJECT'] = PROJECT
print('billing project:', PROJECT)

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project=os.environ['MIMIC_BIGQUERY_PROJECT'])
rows = client.query('SELECT COUNT(*) AS n FROM `physionet-data.mimiciii_clinical.icustays`').result()
print('ICU stays visible:', next(iter(rows)).n)


## 4. Build the cohort

Criteria E1-E10 are applied to the **structured** fields only -- admission type, length of stay,
ICD-9 code sets, a Quan et al. 2005 Charlson index -- never to note vocabulary. The exclusion count at
each criterion is printed and written to the manifest, so the flow diagram is a by-product of
selection rather than something reconstructed afterwards.

`--bq-cache-dir` downloads each table once and reads it locally afterwards. Both commands read all
six tables, so without it the scan -- and the bill -- is paid twice. It changes nothing about the
cohort: the snapshot stays the BigQuery one, so the selection reproduces on any machine, and a cache
written from a different state of `physionet-data` is refused rather than mixed in. It is restricted
data, so it dies with the runtime like everything else under `/content`.

The manifest records a `bigquery-sha256:` snapshot over each table's id, row count and last
modification. That is the BigQuery equivalent of hashing the CSV bytes, and it is deliberately not
interchangeable with one: **the sample is salted with it**, so a cohort you built earlier from the
CSVs will not reproduce here, and `meddial-scr` will refuse to mix the two. Pick one backend and
build the whole pipeline on it.


In [ ]:
!meddial-cohort \
    --bigquery \
    --bq-cache-dir /content/mimic-cache \
    --out /content/meddial-out/cohort \
    --n 200


## 5. Extract the references

One Structured Clinical Reference per selected admission, read off the admission's whole discharge
documentation. Extraction is resumable: a case whose file already exists is skipped, so when the
Colab runtime is recycled mid-run -- and on a long run it will be -- re-running this cell continues
where it stopped, provided the output directory survived. If it did not, copy it off first (step 6).


In [ ]:
!meddial-scr \
    --bigquery \
    --bq-cache-dir /content/mimic-cache \
    --cohort /content/meddial-out/cohort/cohort_private_manifest.json \
    --out /content/meddial-out/references \
    --extractor $MEDDIAL_EXTRACTOR \
    --extractor-family qwen \
    --max-tokens $MEDDIAL_MAX_TOKENS


## 6. Take the output with you

Everything under `/content/meddial-out` is derived from restricted data and disappears with the
runtime. Where it may be stored is a governance decision, not a convenience one, so this notebook
does not mount Drive or upload anything for you. The cell below only packages the directory; run it,
then move the archive somewhere your data use agreement covers.


In [ ]:
!cd /content && zip -qr meddial-out.zip meddial-out && ls -lh /content/meddial-out.zip

# from google.colab import files; files.download('/content/meddial-out.zip')
